# Election Data Analytics Dashboard

**Beginner-friendly IBM SkillsBuild / BharatCares project**

This notebook follows the dashboard sequence taught in the training: **Data -> Information -> Insight -> Decision -> Action**.

> **Data note:** The bundled CSV is synthetic educational data. Replace it with the dataset from your Google Sheet for your final submission. The notebook is descriptive and neutral; it does not predict election outcomes or recommend a political choice.

## 1. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

## 2. Load the dataset

In [ ]:
DATA_PATH = Path('data/election_data.csv')

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print('Rows:', len(df))
print('Columns:', list(df.columns))
print('Constituencies:', df['Constituency'].nunique())

## 3. Data preparation

In [ ]:
df['Turnout_Percent'] = (df['Votes_Cast'] / df['Registered_Voters'] * 100).round(2)
df['Is_Winner'] = df['Rank'].eq(1)

winners = df[df['Is_Winner']].copy()
runner_up = df[df['Rank'].eq(2)].copy()

margins = winners[['Constituency', 'Votes_Received']].merge(
    runner_up[['Constituency', 'Votes_Received']],
    on='Constituency',
    suffixes=('_Winner', '_RunnerUp')
)
margins['Margin'] = margins['Votes_Received_Winner'] - margins['Votes_Received_RunnerUp']

df.describe(include='all').head()

## 4. Executive KPIs

In [ ]:
constituency_level = df.groupby('Constituency').agg(
    Registered_Voters=('Registered_Voters', 'max'),
    Votes_Cast=('Votes_Cast', 'max')
)

constituency_level['Turnout_Percent'] = (
    constituency_level['Votes_Cast'] / constituency_level['Registered_Voters'] * 100
)

kpis = {
    'Constituencies': constituency_level.shape[0],
    'Registered voters': int(constituency_level['Registered_Voters'].sum()),
    'Votes cast': int(constituency_level['Votes_Cast'].sum()),
    'Average turnout %': round(constituency_level['Turnout_Percent'].mean(), 2),
    'Average winning margin': round(margins['Margin'].mean(), 2)
}
pd.Series(kpis)

## 5. Page 1 - Executive Overview

In [ ]:
party_votes = winners.groupby('Party')['Votes_Received'].sum().sort_values(ascending=False)
party_votes.plot(kind='bar')
plt.title('Winning Votes by Party')
plt.xlabel('Party')
plt.ylabel('Votes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
region_turnout = df.groupby('Region').agg(
    Registered=('Registered_Voters', 'sum'),
    Votes=('Votes_Cast', 'sum')
)
region_turnout['Turnout_Percent'] = region_turnout['Votes'] / region_turnout['Registered'] * 100
region_turnout['Turnout_Percent'].sort_values().round(2)

## 6. Page 2 - Party & Region Analysis

In [ ]:
party_region = df.groupby(['Region', 'Party'])['Votes_Received'].sum().unstack(fill_value=0)
party_region.plot(kind='bar', figsize=(10, 5))
plt.title('Votes Received by Region and Party')
plt.xlabel('Region')
plt.ylabel('Votes')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
winners[['Region', 'Constituency', 'Candidate', 'Party', 'Votes_Received']].sort_values(['Region','Constituency']).head(20)

## 7. Page 3 - Turnout & Margin Analysis

In [ ]:
top_margins = margins.sort_values('Margin', ascending=False).head(12).sort_values('Margin')
top_margins.set_index('Constituency')['Margin'].plot(kind='barh')
plt.title('Largest Winning Margins (Top 12 Constituencies)')
plt.xlabel('Vote margin')
plt.ylabel('Constituency')
plt.tight_layout()
plt.show()

In [ ]:
constituency_level['Turnout_Percent'].plot(kind='hist', bins=8)
plt.title('Constituency Turnout Distribution')
plt.xlabel('Turnout (%)')
plt.ylabel('Number of Constituencies')
plt.tight_layout()
plt.show()

## 8. Decision-oriented insights

Use the tables and charts to identify patterns worth investigating further, such as regions with lower turnout or constituencies with relatively small winning margins. These are **data-quality / analytical follow-up areas**, not political recommendations.

### Limitations
- The included dataset is synthetic and should not be presented as real election results.
- Analysis depends on the columns and definitions in the final dataset.
- Turnout and margin are descriptive measures; they do not explain causes or predict future outcomes.